# F1 Podium Prediction - Data Cleansing

**Objective:**
Consolidate 15 CSV files into a single race-driver dataset for model training

Data Range:
- Load: 2006-2023 (2006-2008 for 3-year lookback feature calculation)
- Training: 2009-2022 (with full 3-year historical features)
- Testing: 2023

Output:
- `data/processed/race_driver_records_train.csv` (2009-2022)
- `data/processed/race_driver_records_test.csv` (2023)

In [1]:
 import sys
 sys.path.append('/Users/cathalbailey/Documents/Data Science Institute/Applied-Computer-Science-Project/f1-predictor')
 from src.data_loader import *
 import pandas as pd
 import numpy as np

In [2]:
results = load_results()
races = load_races()
drivers = load_drivers()
constructors = load_constructors()
qualifying = load_qualifying()
driver_standings = load_driver_standings()
constructor_standings = load_constructor_standings()
weather = load_weather()
circuits = load_circuits()

In [3]:
results.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22.0,1,1.0,1,1,10.0,58,1:34:50.616,5690616.0,39.0,2.0,1:27.452,218.300,1
1,2,18,2,2,3.0,5,2.0,2,2,8.0,58,+5.478,5696094.0,41.0,3.0,1:27.739,217.586,1
2,3,18,3,3,7.0,7,3.0,3,3,6.0,58,+8.163,5698779.0,41.0,5.0,1:28.090,216.719,1
3,4,18,4,4,5.0,11,4.0,4,4,5.0,58,+17.181,5707797.0,58.0,7.0,1:28.603,215.464,1
4,5,18,5,1,23.0,3,5.0,5,5,4.0,58,+18.014,5708630.0,43.0,1.0,1:27.418,218.385,1


In [4]:
print("="*60)
print("FILTERING DATA TO 2006 ONWARDS (for 3-year lookback)")
print("="*60)

# Filter races to 2006+ (2006-2008 used for feature calculation, 2009+ for training/test)
races_filtered = races[races['year'] >= 2006].copy()

print(f"Races before filter: {len(races)} ({races['year'].min()}-{races['year'].max()})")
print(f"Races after filter:  {len(races_filtered)} ({races_filtered['year'].min()}-{races_filtered['year'].max()})")

# Get the raceIds for 2006+
valid_race_ids = races_filtered['raceId'].unique()
results_filtered = results[results['raceId'].isin(valid_race_ids)].copy()

print(f"\nResults before filter: {len(results):,} records")
print(f"Results after filter:  {len(results_filtered):,} records")

# Filter other datasets to only include relevant IDs
qualifying_filtered = qualifying[qualifying['raceId'].isin(valid_race_ids)].copy()
driver_standings_filtered = driver_standings[driver_standings['raceId'].isin(valid_race_ids)].copy()
constructor_standings_filtered = constructor_standings[constructor_standings['raceId'].isin(valid_race_ids)].copy()
weather_filtered = weather[weather['raceId'].isin(valid_race_ids)].copy()

print(f"\nQualifying filtered:           {len(qualifying_filtered):,} records")
print(f"Driver standings filtered:     {len(driver_standings_filtered):,} records")
print(f"Constructor standings filtered: {len(constructor_standings_filtered):,} records")
print(f"Weather filtered:              {len(weather_filtered):,} records")

races = races_filtered
results = results_filtered
qualifying = qualifying_filtered
driver_standings = driver_standings_filtered
constructor_standings = constructor_standings_filtered
weather = weather_filtered

print("\n✓ Data filtered to 2006-onwards")
print("  Note: 2006-2008 will be used for feature calculation only")
print("        2009-2022 for training, 2023 for testing")

FILTERING DATA TO 2006 ONWARDS (for 3-year lookback)
Races before filter: 1101 (1950-2023)
Races after filter:  351 (2006-2023)

Results before filter: 26,080 records
Results after filter:  7,235 records

Qualifying filtered:           7,211 records
Driver standings filtered:     7,555 records
Constructor standings filtered: 3,632 records
Weather filtered:              351 records

✓ Data filtered to 2006-onwards
  Note: 2006-2008 will be used for feature calculation only
        2009-2022 for training, 2023 for testing


In [5]:
print("Raw data shapes:")
print(f"Results: {results.shape}")
print(f"Races: {races.shape}")
print(f"Drivers: {drivers.shape}")
print(f"Constructors: {constructors.shape}")
print(f"Qualifying: {qualifying.shape}")
print(f"Driver Standings: {driver_standings.shape}")
print(f"Constructor Standings: {constructor_standings.shape}")
print(f"Weather: {weather.shape}")
print(f"Circuits: {circuits.shape}")

Raw data shapes:
Results: (7235, 18)
Races: (351, 18)
Drivers: (857, 9)
Constructors: (211, 5)
Qualifying: (7211, 9)
Driver Standings: (7555, 7)
Constructor Standings: (3632, 7)
Weather: (351, 4)
Circuits: (77, 9)


In [6]:
print("="*60)
print("STEP 2 & 3: UNIQUE COUNTS & MISMATCH CHECK")
print("="*60)

# Step 2: Unique counts from Results
print("\nUnique counts from Results dataset:")
print(f"  Unique Races:        {results['raceId'].nunique():,}")
print(f"  Unique Drivers:      {results['driverId'].nunique():,}")
print(f"  Unique Constructors: {results['constructorId'].nunique():,}")
print(f"  Total Records:       {len(results):,}")

# Step 3: Mismatch check
print("\n" + "="*60)
print("Reference table unique counts:")
print(f"  Drivers.csv:       {drivers['driverId'].nunique():,} drivers")
print(f"  Constructors.csv:  {constructors['constructorId'].nunique():,} constructors")
print(f"  Races.csv:         {races['raceId'].nunique():,} races")
print(f"  Circuits.csv:      {circuits['circuitId'].nunique():,} circuits")

# Mismatch analysis
print("\n" + "="*60)
print("MISMATCH ANALYSIS:")

drivers_in_results = set(results['driverId'].unique())
drivers_in_master = set(drivers['driverId'].unique())
missing_drivers = drivers_in_results - drivers_in_master
print(f"\n  Drivers in Results but not in Drivers.csv: {len(missing_drivers)}")
if missing_drivers:
    print(f"    Missing IDs: {missing_drivers}")

constructors_in_results = set(results['constructorId'].unique())
constructors_in_master = set(constructors['constructorId'].unique())
missing_constructors = constructors_in_results - constructors_in_master
print(f"  Constructors in Results but not in Constructors.csv: {len(missing_constructors)}")
if missing_constructors:
    print(f"    Missing IDs: {missing_constructors}")

races_in_results = set(results['raceId'].unique())
races_in_master = set(races['raceId'].unique())
missing_races = races_in_results - races_in_master
print(f"  Races in Results but not in Races.csv: {len(missing_races)}")
if missing_races:
    print(f"    Missing IDs: {missing_races}")

# Duplicate check
print("\n" + "="*60)
print("DUPLICATE ID CHECK:")
print(f"  Duplicate driverIds in Drivers.csv:       {drivers['driverId'].duplicated().sum()}")
print(f"  Duplicate constructorIds in Constructors.csv: {constructors['constructorId'].duplicated().sum()}")
print(f"  Duplicate raceIds in Races.csv:           {races['raceId'].duplicated().sum()}")

STEP 2 & 3: UNIQUE COUNTS & MISMATCH CHECK

Unique counts from Results dataset:
  Unique Races:        341
  Unique Drivers:      95
  Unique Constructors: 30
  Total Records:       7,235

Reference table unique counts:
  Drivers.csv:       857 drivers
  Constructors.csv:  211 constructors
  Races.csv:         351 races
  Circuits.csv:      77 circuits

MISMATCH ANALYSIS:

  Drivers in Results but not in Drivers.csv: 0
  Constructors in Results but not in Constructors.csv: 0
  Races in Results but not in Races.csv: 0

DUPLICATE ID CHECK:
  Duplicate driverIds in Drivers.csv:       0
  Duplicate constructorIds in Constructors.csv: 0
  Duplicate raceIds in Races.csv:           0


In [7]:
print("="*60)
print("STEP 4: FREQUENCY DISTRIBUTION BY NATIONALITY")
print("="*60)

# Drivers by nationality
print("\nTop 15 Driver Nationalities:")
driver_nationality = drivers['nationality'].value_counts().head(15)
print(driver_nationality)
print(f"\nTotal nationalities (drivers): {drivers['nationality'].nunique()}")

# Constructors by nationality
print("\n" + "="*60)
print("Top 10 Constructor Nationalities:")
constructor_nationality = constructors['nationality'].value_counts().head(10)
print(constructor_nationality)
print(f"\nTotal nationalities (constructors): {constructors['nationality'].nunique()}")

STEP 4: FREQUENCY DISTRIBUTION BY NATIONALITY

Top 15 Driver Nationalities:
nationality
British          165
American         158
Italian           99
French            73
German            50
Brazilian         32
Argentine         24
Belgian           23
Swiss             23
South African     23
Japanese          20
Dutch             18
Australian        18
Austrian          15
Spanish           15
Name: count, dtype: int64

Total nationalities (drivers): 42

Top 10 Constructor Nationalities:
nationality
British          86
American         39
Italian          29
French           13
German           10
Japanese          5
Swiss             5
South African     3
Dutch             3
Canadian          2
Name: count, dtype: int64

Total nationalities (constructors): 24


In [8]:
# Start with results as base (filtered: 2006-2023 data)
df = results.copy()

# Merge races - add year, round, circuitId, date
df = df.merge(races[['raceId', 'year', 'round', 'circuitId', 'date', 'name']],
              on='raceId',
              how='left',
              suffixes=('', '_race'))

# Merge drivers - add nationality, dob
df = df.merge(drivers[['driverId', 'driverRef', 'forename', 'surname', 'dob', 'nationality']],
              on='driverId',
              how='left')

# Merge constructors - add team info
df = df.merge(constructors[['constructorId', 'constructorRef', 'name', 'nationality']],
              on='constructorId',
              how='left',
              suffixes=('', '_constructor'))

# Merge circuits - add track characteristics
df = df.merge(circuits[['circuitId', 'circuitRef', 'name', 'location', 'country', 'lat', 'lng', 'alt']],
              on='circuitId',
              how='left',
              suffixes=('', '_circuit'))

print(f"After basic merges: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

After basic merges: (7235, 38)
Columns: ['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId', 'year', 'round', 'circuitId', 'date', 'name', 'driverRef', 'forename', 'surname', 'dob', 'nationality', 'constructorRef', 'name_constructor', 'nationality_constructor', 'circuitRef', 'name_circuit', 'location', 'country', 'lat', 'lng', 'alt']


In [9]:
# Merge qualifying - add grid position (starting position)
# Rename qualifying 'position' to 'grid_position' to avoid conflict
qualifying_grid = qualifying[['raceId', 'driverId', 'position']].copy()
qualifying_grid.rename(columns={'position': 'grid_position'}, inplace=True)
df = df.merge(qualifying_grid,
              on=['raceId', 'driverId'],
              how='left')
# Check missing grid positions
print(f"Missing grid positions: {df['grid_position'].isna().sum()} ({df['grid_position'].isna().sum()/len(df)*100:.1f}%)")
print(f"After qualifying merge: {df.shape}")

# Merge weather - expect many nulls (~60% missing)
df = df.merge(weather[['raceId', 'weather_raw']],
              on='raceId',
              how='left')
print(f"Missing weather data: {df['weather_raw'].isna().sum()} ({df['weather_raw'].isna().sum()/len(df)*100:.1f}%)")
print(f"After weather merge: {df.shape}")

Missing grid positions: 24 (0.3%)
After qualifying merge: (7235, 39)
Missing weather data: 22 (0.3%)
After weather merge: (7235, 40)


In [10]:
#Examine wetaher data

print("Weather: Data sample")
df[['raceId', 'year', 'name', 'weather_raw']].head(10)

Weather: Data sample


,raceId,year,name,weather_raw
0,18,2008,Australian Grand Prix,"Clear, dry, air temperature of 39 °C (102 °F)."
1,18,2008,Australian Grand Prix,"Clear, dry, air temperature of 39 °C (102 °F)."
2,18,2008,Australian Grand Prix,"Clear, dry, air temperature of 39 °C (102 °F)."
3,18,2008,Australian Grand Prix,"Clear, dry, air temperature of 39 °C (102 °F)."
4,18,2008,Australian Grand Prix,"Clear, dry, air temperature of 39 °C (102 °F)."
5,18,2008,Australian Grand Prix,"Clear, dry, air temperature of 39 °C (102 °F)."
6,18,2008,Australian Grand Prix,"Clear, dry, air temperature of 39 °C (102 °F)."
7,18,2008,Australian Grand Prix,"Clear, dry, air temperature of 39 °C (102 °F)."
8,18,2008,Australian Grand Prix,"Clear, dry, air temperature of 39 °C (102 °F)."
9,18,2008,Australian Grand Prix,"Clear, dry, air temperature of 39 °C (102 °F)."


In [11]:
# We need previous race standings, not current race standings
# Sort by race date to get chronological order
df = df.sort_values(['driverId', 'date']).reset_index(drop=True)

# Create previous race driver standings (shift by 1 for each driver)
driver_standings_prev = driver_standings.copy()
driver_standings_prev = driver_standings_prev.sort_values(['driverId', 'raceId'])
driver_standings_prev['prev_driver_points'] = driver_standings_prev.groupby('driverId')['points'].shift(1)
driver_standings_prev['prev_driver_position'] = driver_standings_prev.groupby('driverId')['position'].shift(1)

# Merge previous standings
df = df.merge(driver_standings_prev[['raceId', 'driverId', 'prev_driver_points', 'prev_driver_position']],
              on=['raceId', 'driverId'],
              how='left')

# Same for constructor standings
constructor_standings_prev = constructor_standings.copy()
constructor_standings_prev = constructor_standings_prev.sort_values(['constructorId', 'raceId'])
constructor_standings_prev['prev_constructor_points'] = constructor_standings_prev.groupby('constructorId')['points'].shift(1)
constructor_standings_prev['prev_constructor_position'] = constructor_standings_prev.groupby('constructorId')['position'].shift(1)
df = df.merge(constructor_standings_prev[['raceId', 'constructorId', 'prev_constructor_points', 'prev_constructor_position']],
              on=['raceId', 'constructorId'],
              how='left')

print(f"After standings merge: {df.shape}")
print(f"Missing prev driver standings: {df['prev_driver_position'].isna().sum()}")
print(f"Missing prev constructor standings: {df['prev_constructor_position'].isna().sum()}")

After standings merge: (7235, 44)
Missing prev driver standings: 145
Missing prev constructor standings: 78


In [12]:
df['is_podium'] = (df['position'] <= 3).astype(int)

# Check distribution
print(f"Podium rate: {df['is_podium'].mean()*100:.1f}%")
print(f"Podium count: {df['is_podium'].sum()}")
print(f"\nTarget distribution:")
print(df['is_podium'].value_counts())

Podium rate: 14.1%
Podium count: 1023

Target distribution:
is_podium
0    6212
1    1023
Name: count, dtype: int64


In [13]:
print("="*60)
print("STEP 7: CONSTRUCTOR ANALYSIS - TOP 3 FINISHERS")
print("="*60)

# Group by constructor
constructor_podiums = df.groupby('name_constructor').agg({
    'is_podium': ['sum', 'mean', 'count']
}).round(3)
constructor_podiums.columns = ['Total_Podiums', 'Podium_Rate', 'Total_Races']
constructor_podiums = constructor_podiums.sort_values('Total_Podiums', ascending=False)

print("\nTop 15 Constructors by Podium Count (2009+):")
print(constructor_podiums.head(15))

print("\n" + "="*60)
print("STEP 8: DRIVER ANALYSIS - TOP 3 FINISHERS")
print("="*60)

# Combine driver name
df['driver_name'] = df['forename'] + ' ' + df['surname']

# Group by driver
driver_podiums = df.groupby('driver_name').agg({
    'is_podium': ['sum', 'mean', 'count']
}).round(3)
driver_podiums.columns = ['Total_Podiums', 'Podium_Rate', 'Total_Races']
driver_podiums = driver_podiums.sort_values('Total_Podiums', ascending=False)

print("\nTop 20 Drivers by Podium Count (2009+):")
print(driver_podiums.head(20))

# Minimum 30 races filter
print("\n" + "="*60)
print("Top 15 Drivers by Podium Rate (min 30 races):")
experienced_drivers = driver_podiums[driver_podiums['Total_Races'] >= 30]
print(experienced_drivers.sort_values('Podium_Rate', ascending=False).head(15))

STEP 7: CONSTRUCTOR ANALYSIS - TOP 3 FINISHERS

Top 15 Constructors by Podium Count (2009+):
                  Total_Podiums  Podium_Rate  Total_Races
name_constructor                                         
Mercedes                    269        0.496          542
Red Bull                    253        0.371          682
Ferrari                     239        0.350          682
McLaren                     111        0.163          682
Renault                      33        0.079          416
Lotus F1                     25        0.162          154
Williams                     20        0.029          682
BMW Sauber                   17        0.121          140
Brawn                        15        0.441           34
Toyota                        8        0.057          140
Aston Martin                  7        0.062          112
Force India                   6        0.014          424
Sauber                        4        0.011          354
Racing Point                  4      

In [14]:
# 1. Grid position missing - impute with max_grid + 1 (started from pit lane)
max_grid = df['grid_position'].max()
df['grid_position'] = df['grid_position'].fillna(max_grid + 1)

print(f"Grid positions imputed with {max_grid + 1} for missing values")

# 2. Weather missing - create 'unknown' category
df['weather_raw'] = df['weather_raw'].fillna('unknown')

# 3. Standings missing (rookie drivers/constructors) - fill with 0
df['prev_driver_points'] = df['prev_driver_points'].fillna(0)
df['prev_driver_position'] = df['prev_driver_position'].fillna(99) # 99 not in championship
df['prev_constructor_points'] = df['prev_constructor_points'].fillna(0)
df['prev_constructor_position'] = df['prev_constructor_position'].fillna(99)


# Check remaining nulls
print(f"\nRemaining nulls per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Grid positions imputed with 25.0 for missing values

Remaining nulls per column:
position           1316
time               3654
milliseconds       3654
fastestLap          321
rank                105
fastestLapTime      321
fastestLapSpeed     321
alt                  60
dtype: int64


In [15]:
# 1. Check for duplicate (raceId, driverId) combinations
duplicates = df.duplicated(subset=['raceId', 'driverId']).sum()
print(f"Duplicate (raceId, driverId) pairs: {duplicates}")

# 2. Verify podium rate is reasonable (~15% expected)
podium_rate = df['is_podium'].mean()
print(f"Podium rate: {podium_rate*100:.2f}% (expected ~15%)")

# 3. Check position values are valid
print(f"\nPosition value range: {df['position'].min()} to {df['position'].max()}")
print(f"Unique positions: {df['position'].nunique()}")

# 4. Verify no \N values remain (all should be NaN or filled)
for col in df.columns:
    if df[col].dtype == 'object':
        if (df[col] == '\\N').any():
            print(f"WARNING: \\N found in {col}")
            
print("\nData validation complete")

Duplicate (raceId, driverId) pairs: 0
Podium rate: 14.14% (expected ~15%)

Position value range: 1.0 to 24.0
Unique positions: 24

Data validation complete


In [16]:
# Split into train/test
# Note: This keeps 2006-2008 data in the saved files for feature engineering
# Feature engineering notebook will filter to 2009+ for final modeling dataset
train = df[df['year'] < 2023].copy()  # 2006-2022
test = df[df['year'] == 2023].copy()   # 2023

print(f"Training set: {train.shape} ({train['year'].min()}-{train['year'].max()})")
print(f"Test set: {test.shape} (year {test['year'].unique()})")
print(f"\nTrain podium rate: {train['is_podium'].mean()*100:.2f}%")
print(f"Test podium rate: {test['is_podium'].mean()*100:.2f}%")

print("\nNote: 2006-2008 data included for 3-year lookback feature calculation")
print("      Final modeling dataset will be 2009-2022 (train) and 2023 (test)")

Training set: (6995, 46) (2006-2022)
Test set: (240, 46) (year [2023])

Train podium rate: 14.11%
Test podium rate: 15.00%

Note: 2006-2008 data included for 3-year lookback feature calculation
      Final modeling dataset will be 2009-2022 (train) and 2023 (test)


In [17]:
# Save to processed folder
train.to_csv('../data/processed/race_driver_records_train.csv', index=False)
test.to_csv('../data/processed/race_driver_records_test.csv', index=False)

print("✓ Files saved:")
print("  - data/processed/race_driver_records_train.csv")
print("  - data/processed/race_driver_records_test.csv")

✓ Files saved:
  - data/processed/race_driver_records_train.csv
  - data/processed/race_driver_records_test.csv


In [18]:
print("="*60)
print("FINAL DATASET SUMMARY")
print("="*60)
print(f"\nTotal features: {train.shape[1]}")
print(f"\nColumn names:")
print(train.columns.tolist())
print(f"\nSample rows:")
print(train.head(3))
print(f"\nData types:")
print(train.dtypes.value_counts())

FINAL DATASET SUMMARY

Total features: 46

Column names:
['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId', 'year', 'round', 'circuitId', 'date', 'name', 'driverRef', 'forename', 'surname', 'dob', 'nationality', 'constructorRef', 'name_constructor', 'nationality_constructor', 'circuitRef', 'name_circuit', 'location', 'country', 'lat', 'lng', 'alt', 'grid_position', 'weather_raw', 'prev_driver_points', 'prev_driver_position', 'prev_constructor_points', 'prev_constructor_position', 'is_podium', 'driver_name']

Sample rows:
   resultId  raceId  driverId  constructorId  number  grid  position  \
0       371      36         1              1     2.0     4       3.0   
1       392      37         1              1     2.0     4       2.0   
2       414      38         1              1     2.0     2       2.0   

  posit

In [19]:
train.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,lng,alt,grid_position,weather_raw,prev_driver_points,prev_driver_position,prev_constructor_points,prev_constructor_position,is_podium,driver_name
0,371,36,1,1,2.0,4,3.0,3,3,6.0,...,144.96800,10.0,4.0,Clear with temperatures reaching up to 21 °C (...,98.0,1.0,151.0,2.0,1,Lewis Hamilton
1,392,37,1,1,2.0,4,2.0,2,2,8.0,...,101.73800,18.0,4.0,Sunny[1],6.0,3.0,14.0,11.0,1,Lewis Hamilton
2,414,38,1,1,2.0,2,2.0,2,2,8.0,...,50.51060,7.0,2.0,"Sunny, dry, windy",14.0,3.0,32.0,11.0,1,Lewis Hamilton
3,436,39,1,1,2.0,4,2.0,2,2,8.0,...,2.26111,109.0,4.0,Sunny and warm,22.0,3.0,44.0,11.0,1,Lewis Hamilton
4,458,40,1,1,2.0,2,2.0,2,2,8.0,...,7.42056,7.0,2.0,"Cloudy, air temp 24–27 °C (75–81 °F), track te...",30.0,1.0,58.0,11.0,1,Lewis Hamilton
